# Slowly Changing Dimensions (SCD) Implementation

## SCD Type 1
- **Overwrites** old data with new data
- No history is maintained
- Simple and uses less storage
- Use case: Correcting errors or when history is not needed

## SCD Type 2
- **Maintains full history** of changes
- Creates new records for changes with versioning
- Typical columns: `is_current`, `effective_date`, `end_date`, `version`
- Use case: Audit trails, historical analysis, compliance

In [0]:
-- Create a sample source table with customer data
CREATE OR REPLACE TABLE customer_source (
  customer_id INT,
  name STRING,
  email STRING,
  city STRING,
  phone STRING
);

INSERT INTO customer_source VALUES
  (1, 'John Doe', 'john@email.com', 'New York', '555-0101'),
  (2, 'Jane Smith', 'jane@email.com', 'Los Angeles', '555-0102'),
  (3, 'Bob Johnson', 'bob@email.com', 'Chicago', '555-0103');

SELECT * FROM customer_source;

In [0]:
-- Create target table for SCD Type 1
CREATE OR REPLACE TABLE customer_type1 (
  customer_id INT,
  name STRING,
  email STRING,
  city STRING,
  phone STRING
);

INSERT INTO customer_type1 VALUES
  (1, 'John Doe', 'john.old@email.com', 'Boston', '555-0001'),
  (2, 'Jane Smith', 'jane@email.com', 'Los Angeles', '555-0102');

SELECT * FROM customer_type1;

In [0]:
-- SCD Type 1: Overwrite changes (no history)
MERGE INTO customer_type1 AS target
USING customer_source AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET 
    target.name = source.name,
    target.email = source.email,
    target.city = source.city,
    target.phone = source.phone
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, email, city, phone)
  VALUES (source.customer_id, source.name, source.email, source.city, source.phone);

-- View results
SELECT * FROM customer_type1;

In [0]:
-- Create target table for SCD Type 2 with history columns
CREATE OR REPLACE TABLE customer_type2 (
  customer_id INT,
  name STRING,
  email STRING,
  city STRING,
  phone STRING,
  is_current BOOLEAN,
  effective_date DATE,
  end_date DATE,
  version INT
);

INSERT INTO customer_type2 VALUES
  (1, 'John Doe', 'john.old@email.com', 'Boston', '555-0001', true, '2024-01-01', '9999-12-31', 1),
  (2, 'Jane Smith', 'jane@email.com', 'Los Angeles', '555-0102', true, '2024-01-01', '9999-12-31', 1);

SELECT * FROM customer_type2;

In [0]:
SELECT * FROM customer_source

In [0]:
  -- Part 1: Records to expire (changed records) - these will MATCH
  SELECT 
    s.customer_id,
    t.name,
    t.email,
    t.city,
    t.phone,
    'EXPIRE' as operation_type,
    CAST(NULL AS INT) as new_version
  FROM customer_source s
  INNER JOIN customer_type2 t 
    ON s.customer_id = t.customer_id 
    AND t.is_current = true
  WHERE t.name != s.name 
     OR t.email != s.email 
     OR t.city != s.city 
     OR t.phone != s.phone

In [0]:
  -- Part 2: New versions for changed records - these will NOT MATCH
  SELECT 
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.phone,
    'INSERT_CHANGED' as operation_type,
    (SELECT MAX(version) FROM customer_type2 WHERE customer_id = s.customer_id) + 1 as new_version
  FROM customer_source s
  INNER JOIN customer_type2 t 
    ON s.customer_id = t.customer_id 
    AND t.is_current = true
  WHERE t.name != s.name 
     OR t.email != s.email 
     OR t.city != s.city 
     OR t.phone != s.phone

In [0]:
 SELECT 
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.phone,
    'INSERT_NEW' as operation_type,
    1 as new_version
  FROM customer_source s
  WHERE NOT EXISTS (
    SELECT 1 FROM customer_type2 t
    WHERE t.customer_id = s.customer_id
  )

In [0]:
  -- Part 1: Records to expire (changed records) - these will MATCH
  SELECT 
    s.customer_id,
    t.name,
    t.email,
    t.city,
    t.phone,
    'EXPIRE' as operation_type,
    CAST(NULL AS INT) as new_version
  FROM customer_source s
  INNER JOIN customer_type2 t 
    ON s.customer_id = t.customer_id 
    AND t.is_current = true
  WHERE t.name != s.name 
     OR t.email != s.email 
     OR t.city != s.city 
     OR t.phone != s.phone
  
  UNION ALL
  
  -- Part 2: New versions for changed records - these will NOT MATCH
  SELECT 
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.phone,
    'INSERT_CHANGED' as operation_type,
    (SELECT MAX(version) FROM customer_type2 WHERE customer_id = s.customer_id) + 1 as new_version
  FROM customer_source s
  INNER JOIN customer_type2 t 
    ON s.customer_id = t.customer_id 
    AND t.is_current = true
  WHERE t.name != s.name 
     OR t.email != s.email 
     OR t.city != s.city 
     OR t.phone != s.phone
  
  UNION ALL
  
  -- Part 3: Brand new customers - these will NOT MATCH
  SELECT 
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.phone,
    'INSERT_NEW' as operation_type,
    1 as new_version
  FROM customer_source s
  WHERE NOT EXISTS (
    SELECT 1 FROM customer_type2 t
    WHERE t.customer_id = s.customer_id
  )

In [0]:
-- SCD Type 2: Single MERGE statement handling both expire and insert operations
MERGE INTO customer_type2 AS target
USING (
  -- Part 1: Records to expire (changed records) - these will MATCH
  SELECT 
    s.customer_id,
    t.name,
    t.email,
    t.city,
    t.phone,
    'EXPIRE' as operation_type,
    CAST(NULL AS INT) as new_version
  FROM customer_source s
  INNER JOIN customer_type2 t 
    ON s.customer_id = t.customer_id 
    AND t.is_current = true
  WHERE t.name != s.name 
     OR t.email != s.email 
     OR t.city != s.city 
     OR t.phone != s.phone
  
  UNION ALL
  
  -- Part 2: New versions for changed records - these will NOT MATCH
  SELECT 
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.phone,
    'INSERT_CHANGED' as operation_type,
    (SELECT MAX(version) FROM customer_type2 WHERE customer_id = s.customer_id) + 1 as new_version
  FROM customer_source s
  INNER JOIN customer_type2 t 
    ON s.customer_id = t.customer_id 
    AND t.is_current = true
  WHERE t.name != s.name 
     OR t.email != s.email 
     OR t.city != s.city 
     OR t.phone != s.phone
  
  UNION ALL
  
  -- Part 3: Brand new customers - these will NOT MATCH
  SELECT 
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.phone,
    'INSERT_NEW' as operation_type,
    1 as new_version
  FROM customer_source s
  WHERE NOT EXISTS (
    SELECT 1 FROM customer_type2 t
    WHERE t.customer_id = s.customer_id
  )
) AS source
ON target.customer_id = source.customer_id 
   AND target.is_current = true 
   AND source.operation_type = 'EXPIRE'
WHEN MATCHED THEN
  UPDATE SET 
    target.is_current = false,
    target.end_date = current_date()
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, email, city, phone, is_current, effective_date, end_date, version)
  VALUES (source.customer_id, source.name, source.email, source.city, source.phone, 
          true, current_date(), cast('9999-12-31' AS date), source.new_version);

-- View all records with history
SELECT * FROM customer_type2 ORDER BY customer_id, version;